In [0]:
from pyspark.sql.functions import *

# Azure Event Hub Configuration
event_hub_namespace = "hospital-analytics-namespace.servicebus.windows.net"
event_hub_name="hospital-analytics-eh"  
event_hub_conn_str = dbutils.secrets.get(scope = "hospitalanalyticsvaultscope", key = "eventhub-connection")


kafka_options = {
    'kafka.bootstrap.servers': f"{event_hub_namespace}:9093",
    'subscribe': event_hub_name,
    'kafka.security.protocol': 'SASL_SSL',
    'kafka.sasl.mechanism': 'PLAIN',
    'kafka.sasl.jaas.config': f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{event_hub_conn_str}";',
    'startingOffsets': 'latest',
    'failOnDataLoss': 'false'
}

#Read from eventhub
raw_df = (spark.readStream
        .format("Kafka")
        .options(**kafka_options)
        .load()
        )

# Cast data to JSON
json_df = raw_df.selectExpr("CAST(value AS STRING) as raw_json")

# ADLS configuration
storage_key = dbutils.secrets.get(
    scope="hospitalanalyticsvaultscope",
    key="storage-conn"
).strip()

spark.conf.set(
    "fs.azure.account.key.hospitalstorageadls.dfs.core.windows.net",
    storage_key
)

# Bronze path
bronze_path = "abfss://bronze@hospitalstorageadls.dfs.core.windows.net/patient_flow"

# Checkpoint path
checkpoint_path = "abfss://bronze@hospitalstorageadls.dfs.core.windows.net/_checkpoints/patient_flow"

#Write stream to bronze
(
    json_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .start(bronze_path)
)

In [0]:
display(spark.read.format("delta").load(bronze_path))

raw_json
"{""patient_id"": ""d8f37a6e-bda1-49e8-9ff1-90aa23ac1bdb"", ""gender"": ""Male"", ""age"": 98, ""department"": ""Emergency"", ""admission_time"": ""2026-11-20T03:19:12.105655"", ""discharge_time"": ""2026-11-20T04:19:12.105655"", ""bed_id"": 321, ""hospital_id"": 1}"
"{""patient_id"": ""edf342d1-7f9f-4e80-b740-bb1a4632d43d"", ""gender"": ""Female"", ""age"": 96, ""department"": ""Surgery"", ""admission_time"": ""2026-11-21T05:19:13.280807"", ""discharge_time"": ""2026-11-21T22:19:13.280807"", ""bed_id"": 249, ""hospital_id"": 1}"
"{""patient_id"": ""da003da3-4de4-4dbc-b22e-1f91c4566fed"", ""gender"": ""Female"", ""age"": 61, ""department"": ""ICU"", ""admission_time"": ""2026-11-21T13:19:14.283449"", ""discharge_time"": ""2026-11-21T21:19:14.283449"", ""bed_id"": 457, ""hospital_id"": 1}"
"{""patient_id"": ""49b1537d-025e-44d1-b492-2650530b9031"", ""gender"": ""Male"", ""age"": 25, ""department"": ""Oncology"", ""admission_time"": ""2026-11-20T10:19:15.285283"", ""discharge_time"": ""2026-11-22T12:19:15.285283"", ""bed_id"": 55, ""hospital_id"": 5}"
"{""patient_id"": ""ac732a3e-8f87-471b-9c57-c760afb65d38"", ""gender"": ""Female"", ""age"": 24, ""department"": ""Surgery"", ""admission_time"": ""2026-11-19T09:19:16.287180"", ""discharge_time"": ""2026-11-22T03:19:16.287180"", ""bed_id"": 226, ""hospital_id"": 3}"
"{""patient_id"": ""e37635b9-f983-481a-9674-fe56d1f94c84"", ""gender"": ""Male"", ""age"": 2, ""department"": ""Surgery"", ""admission_time"": ""2026-11-22T07:19:17.290139"", ""discharge_time"": ""2026-11-23T20:19:17.290139"", ""bed_id"": 331, ""hospital_id"": 1}"
"{""patient_id"": ""50e6273d-1518-4a2a-9267-26a139d1415b"", ""gender"": ""Female"", ""age"": 53, ""department"": ""Emergency"", ""admission_time"": ""2026-11-21T04:19:18.412365"", ""discharge_time"": ""2026-11-23T00:19:18.412365"", ""bed_id"": 395, ""hospital_id"": 3}"
"{""patient_id"": ""5e5f04dd-af8a-4b6a-84df-684409a31208"", ""gender"": ""Female"", ""age"": 3, ""department"": ""Pediatrics"", ""admission_time"": ""2026-11-19T15:19:19.416112"", ""discharge_time"": ""2026-11-20T07:19:19.416112"", ""bed_id"": 493, ""hospital_id"": 2}"
"{""patient_id"": ""b8671fb6-62e9-447a-bb71-ac52b100902c"", ""gender"": ""Male"", ""age"": 64, ""department"": ""Oncology"", ""admission_time"": ""2026-11-19T19:19:20.419575"", ""discharge_time"": ""2026-11-20T08:19:20.419575"", ""bed_id"": 395, ""hospital_id"": 5}"
"{""patient_id"": ""258d444b-98d6-4a83-a7c5-204e2ec5fc06"", ""gender"": ""Female"", ""age"": 26, ""department"": ""Maternity"", ""admission_time"": ""2026-11-21T05:19:21.420893"", ""discharge_time"": ""2026-11-23T15:19:21.420893"", ""bed_id"": 188, ""hospital_id"": 2}"


## Validation 

In [0]:
import base64

storage_key = dbutils.secrets.get(
    scope="hospitalanalyticsvaultscope",
    key="storage-conn"
).strip()

print("Secret retrieved successfully")
print("Key length:", len(storage_key))

try:
    decoded_key = base64.b64decode(storage_key, validate=True)
    print("Base64 validation: PASSED")
    print("Decoded key length:", len(decoded_key))
except Exception as e:
    print("Base64 validation: FAILED")
    print("Error:", str(e))

Secret retrieved successfully
Key length: 88
Base64 validation: PASSED
Decoded key length: 64


In [0]:
storage_key = dbutils.secrets.get(
    scope="hospitalanalyticsvaultscope",
    key="storage-conn"
).strip()

spark.conf.set(
    "fs.azure.account.key.hospitalstorageadls.dfs.core.windows.net",
    storage_key
)

# Test storage connectivity
display(
    dbutils.fs.ls(
        "abfss://bronze@hospitalstorageadls.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://bronze@hospitalstorageadls.dfs.core.windows.net/_checkpoints/,_checkpoints/,0,1795083993000
